# LWM Fabricator — Full Live Demo on Google Colab (Free T4 GPU)
## Capability Fabrication via Latent World Models

**This notebook:**
1. Installs Ollama + pulls qwen3:4b (reflex) and qwen3:14b (judge)
2. Clones the lwm-fabricator repo from GitHub
3. Runs all 9 agentic layers with live LLM debate
4. Demonstrates proactive simulation, MCP protocol, and RL credit assignment

**Runtime:** Select GPU (T4) from Runtime → Change runtime type

## Step 1: Install Ollama & Pull Models

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.ai/install.sh | sh

# Start Ollama server in background
import subprocess
import time

proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)
print('Ollama server started')

In [ ]:
# Pull models (this takes a few minutes)
!ollama pull qwen3:4b
!ollama pull qwen3:14b

# Verify
!ollama list

## Step 2: Clone lwm-fabricator from GitHub

```bash
!git clone https://github.com/Aravindh-dev12/lwm-fabricator-model.git
```

In [ ]:
import sys, os
os.makedirs('data', exist_ok=True)

# Clone from GitHub
!git clone https://github.com/Aravindh-dev12/lwm-fabricator-model.git
sys.path.insert(0, 'lwm-fabricator-model')

# Install dependencies
!pip install torch numpy requests

from lwm_fab.kernel import LWMFabricator
from lwm_fab.ollama_integration import OllamaClient, HybridLLMClient
from lwm_fab.world_model import LeWMWorldModel
from lwm_fab.mcp_agent import MCPAgentRegistry, create_mcp_agents_for_domains, MCPPrompt
from lwm_fab.domain_registry import DOMAINS
from lwm_fab.models import DAGNode, ActionType, ActionVerb, ConsentLevel
import torch, json, time

print(f'Torch: {torch.__version__}')
print(f'Domains: {len(DOMAINS)}')

# Verify Ollama is running with models
oc = OllamaClient()
status = oc.status()
print(f'Ollama: {status}')
assert status['available'], 'Ollama not running! Check server startup.'

## Step 3: Initialize Kernel with Live LLM

In [ ]:
kernel = LWMFabricator(
    mode='dry_run',
    db_path='data/lwm_fab.db',
    ollama_url='http://localhost:11434',
    judge_model='qwen3:14b',
    reflex_model='qwen3:4b',
)

# Verify live LLM
stats = kernel._system_stats()
print('System Stats:')
for layer, s in stats.items():
    print(f'  {layer}: {s}')

ollama_status = stats.get('ollama', {})
print(f'\nLLM Backend: {ollama_status.get("active_backend", "ollama")}')
print(f'Models: {ollama_status.get("models", [])}')

## Step 4: Live MACI Debate Demo
Watch qwen3:4b argue pro/con and qwen3:14b judge in real-time

In [ ]:
from lwm_fab.ollama_integration import OllamaDebateProtocol

debate = OllamaDebateProtocol(kernel.ollama.ollama if hasattr(kernel.ollama, 'ollama') else kernel.ollama)

actions = [
    ('shell:execute — rm -rf /tmp/cache', 'elevated consent, destructive verb'),
    ('file:write — config.yaml', 'standard consent, non-destructive'),
    ('network:request — POST https://api.stripe.com/charges', 'never consent, financial action'),
]

for action, context in actions:
    print(f'\n{"="*70}')
    print(f'Action: {action}')
    print(f'Context: {context}')
    print(f'{"="*70}')
    
    result = debate.debate(action, context)
    print(f'\nProponent (qwen3:4b):')
    print(f'  {result["proponent"][:300]}')
    print(f'\nOpponent (qwen3:4b):')
    print(f'  {result["opponent"][:300]}')
    print(f'\nJudge (qwen3:14b):')
    print(f'  {result["judge"][:400]}')
    print(f'\nVerdict: {result["verdict"]} (confidence: {result["confidence"]})')

## Step 5: Full End-to-End Pipeline with Live LLM

In [ ]:
intent = 'Build a marketing landing page and set up automated email campaigns for our SaaS product'
print(f'Intent: "{intent}"')
print(f'\nRunning full 9-layer pipeline with live LLM...\n')

result = kernel.process_intent(intent)

print(f'Run ID: {result["run_id"]}')
print(f'Status: {result["final_status"]}')
print(f'Domains: {result["matched_domains"]}')
print(f'Nodes: {len(result["dag"]["nodes"])}')

print(f'\nPipeline:')
for step in result['pipeline_log']:
    e = step.get('elapsed_ms', '—')
    print(f'  {step["step"]:<25} {e}ms' if isinstance(e, (int, float)) else f'  {step["step"]:<25} {e}')

print(f'\nNode Results:')
for nr in result['node_results']:
    icon = '✓' if nr.get('status') == 'success' else '⏸' if nr.get('status') == 'paused' else '✗'
    sg = f' [gate: {nr["safety_gate"]["verdict"]}]' if 'safety_gate' in nr else ''
    code = f' [code: {nr["generated_code"][:80]}...]' if 'generated_code' in nr else ''
    print(f'  {icon} {nr["node_id"]}: {nr.get("status")} — {nr.get("detail", "")[:50]}{sg}{code}')

print(f'\nRL Bottleneck: {result["rl_bottleneck"]}')

## Step 6: LeWM Transformer + CEM Planning Benchmark

In [ ]:
wm = kernel.world_model
print(f'LeWM Transformer Predictor:')
print(f'  Depth: {wm.predictor_depth}, Heads: {wm.predictor_heads}, MLP: {wm.predictor_mlp_dim}')
print(f'  History: {wm.history_size}, Latent dim: {wm.latent_dim}')
print(f'  Total params: {sum(p.numel() for p in wm.predictor.parameters()):,}')

# CEM planning
z_0 = torch.randn(wm.latent_dim)
z_goal = torch.randn(wm.latent_dim)
best_seq, cost, elapsed = wm.cem_plan(z_0, z_goal)

print(f'\nCEM Planning:')
print(f'  Time: {elapsed:.4f}s')
print(f'  Cost: {cost:.4f}')
print(f'  Candidates: {wm.cem_samples * wm.cem_iterations}')
print(f'  vs LLM text prediction (qwen3:14b): ~47.3s')
print(f'  Speedup: {47.3 / max(elapsed, 0.001):.1f}x')

## Step 7: Proactive Simulation (1000 Scenarios)

In [ ]:
psim = kernel.run_proactive_simulation([0.5]*64, [0.8]*64)
print('Proactive Simulation (1000 scenarios):')
for k, v in psim.items():
    print(f'  {k}: {v}')

print(f'\nPer paper §4.3:')
print(f'  >95% confidence → automatic action')
print(f'  60-80% confidence → prompt user')
print(f'  <60% confidence → hold')

## Step 8: MCP Protocol Demo

In [ ]:
agent = kernel.mcp_registry.get_by_domain('data_analysis')
print(f'MCP Agent: {agent.name}')
print(f'  Tools: {len(agent.tools)}')
print(f'  Resources: {len(agent.resources)}')
print(f'  Prompts: {len(agent.prompts)}')

# MCP protocol handler
print(f'\nMCP Protocol — tools/list:')
print(f'  {agent.protocol.handle("tools/list", {})}')

print(f'\nMCP Protocol — prompts/list:')
print(f'  {agent.protocol.handle("prompts/list", {})}')

print(f'\nMCP Protocol — prompts/invoke:')
result = agent.protocol.handle('prompts/invoke', {
    'name': 'fabricate_data_analysis',
    'arguments': {'intent': 'Analyze sales data and create a report'}
})
print(f'  {result}')

print(f'\nMCP Protocol — agent/info:')
print(f'  {agent.protocol.handle("agent/info", {})}')

## Step 9: Run Multiple Intents + Show Learning

In [ ]:
intents = [
    'Analyze our CSV dataset and generate a statistical report with charts',
    'Deploy a Python web app to AWS and configure auto-scaling',
    'Scrape competitor pricing data and build a comparison dashboard',
]

for i, intent in enumerate(intents, 1):
    print(f'\n--- Intent {i}: "{intent}" ---')
    r = kernel.process_intent(intent)
    print(f'  Status: {r["final_status"]}, Domains: {r["matched_domains"]}, Nodes: {len(r["dag"]["nodes"])}')

# Show telemetry learning
print(f'\nTelemetry Router (after {i+1} runs):')
for arm in kernel._system_stats().get('telemetry', {}).get('bandit', {}).get('arms', []):
    if arm['pulls'] > 0:
        print(f'  {arm["name"]:<30} pulls={arm["pulls"]:<5} avg_reward={arm["avg_reward"]:.4f}')

# Show memory
print(f'\nMemory: {kernel._system_stats().get("memory", {})}')

kernel.close()
print('\nDone! All 9 layers ran with live LLM.')